##Env Setup, imports and data load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -n "/content/drive/My Drive/med_dataset.zip" -d "/content/med_dataset"

In [ ]:
!pip install transformers torch datasets
!pip install datasets huggingface_hub
!pip install fvcore

In [ ]:
from transformers import ConvNextImageProcessor, ConvNextForImageClassification
import torch
import time
import random
import numpy as np
import requests
from PIL import Image
from datasets import load_dataset
import copy
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


#doing this so that we always run "random shuffling" to be the same for testing
#consistency

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
from huggingface_hub import login
login()

In [ ]:
processor = ConvNextImageProcessor.from_pretrained("facebook/convnext-large-224-22k-1k")
model = ConvNextForImageClassification.from_pretrained("facebook/convnext-large-224-22k-1k")

196233410 Parameters

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("/content/med_dataset/med_dataset")
print(dataset)
print(dataset["train"][0])

In [ ]:
def transform(batch):
  images = [img.convert("RGB") for img in batch["image"]]   #Convert to RGB - xrays are grayscale but need to align to pretrained model input channels

  pixel_values = processor(images, return_tensors="pt")["pixel_values"]   #Process into tensors
  return {
    "pixel_values": pixel_values,
    "label": batch["label"],
    }

dataset = dataset.with_transform(transform)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset["train"], batch_size=32, shuffle=True) #reduce batches for weaker GPUs, currently using L4
test_loader = DataLoader(dataset["test"], batch_size=32)

Example loader return

In [ ]:
for batch in train_loader:
  print(batch["pixel_values"].shape, batch["label"])
  break

##Implementation

Amend classifier head to have 2 output channels

In [ ]:
import torch.nn as nn
model.classifier = nn.Linear(model.classifier.in_features, 2)
model.config.num_labels = 2

Model init and train loop

In [ ]:
from torch import nn
from torch.optim import AdamW
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
!ls /content/

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
  model.train()
  model.to(device)

  total_loss = 0.0

  train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

  for batch in train_loop:
    pixel_values = batch["pixel_values"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()

    out = model(pixel_values=pixel_values, labels=labels)
    loss = out["loss"]  #weighted sum

    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    train_loop.set_postfix({"loss": loss.item()})

  print(f"Epoch {epoch+1}: avg train loss = {total_loss / len(train_loader):.4f}")


In [ ]:
def evaluate_model(model, dataloader, device, label="", cast_half=False):
  model.to(device)
  model.eval()

  correct = 0
  total = 0
  total_infer_time = 0.0

  with torch.no_grad():
    for batch in dataloader:
      pixel_values = batch["pixel_values"].to(device, non_blocking=True)
      if cast_half:
        pixel_values = pixel_values.half()
      labels = batch["label"]

      # if device.type == "cuda":
      torch.cuda.synchronize()

      start = time.time()
      outputs = model(pixel_values=pixel_values)
      # if device.type == "cuda":
      torch.cuda.synchronize()

      elapsed = time.time() - start
      total_infer_time += elapsed

      preds = outputs.logits.argmax(dim=-1).cpu()
      correct += (preds == labels).sum().item()
      total += labels.size(0)

  accuracy = (correct / total) * 100
  avg_batch_time = total_infer_time / len(dataloader)
  avg_sample_time = total_infer_time / total

  label_str = f"{label} " if label else ""
  print(
    f"{label_str}accuracy: {accuracy:.4f}%, "
    f"total_infer_time: {total_infer_time:.4f} s, "
    f"avg_per_batch: {avg_batch_time*1000:.2f} ms, "
    f"avg_per_sample: {avg_sample_time*1000:.4f} ms"
    )

  return accuracy, total_infer_time

baseline_acc, baseline_time = evaluate_model(model, test_loader, device, label="FP32")

In [ ]:
def tensorwise_quantize(x: torch.Tensor, num_bits: int = 8) -> torch.Tensor:
  minVal = torch.min(x)
  maxVal = torch.max(x)

  if (maxVal - minVal).abs() < 1e-8:  #avoid div by 0
    return x.clone()

  qmin = 0
  qmax = (1 << num_bits) - 1
  scale = (maxVal - minVal) / float(qmax - qmin)
  xClipped = torch.clamp(x, min=minVal, max=maxVal)
  xInt = torch.round((xClipped - minVal) / scale)
  xInt = torch.clamp(xInt, qmin, qmax)
  x_quant = xInt * scale + minVal

  return x_quant

def apply_quantization(model: nn.Module, num_bits: int = 8):
  print(f"\n--- Applying {num_bits}-bit Quantization ---")

  q_model = copy.deepcopy(model)    #maintain original
  q_model.eval()

  count = 0

  for name, module in q_model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
      with torch.no_grad():
        module.weight.data = tensorwise_quantize(module.weight.data, num_bits)
        if module.bias is not None:
            module.bias.data = tensorwise_quantize(module.bias.data, num_bits)
      count += 1

  print(f"Quantized {count} layers to {num_bits}-bit precision.")
  return q_model

quantized_model = apply_quantization(model, num_bits=8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
quantized_model.to(device)

print("\n--- Evaluation: 8-Bit Quantized Model ---")

q_acc, q_time = evaluate_model(
  quantized_model,
  test_loader,
  device,
  label="INT8"
  )

def model_save_size(model, num_bits):
  total_params = sum(p.numel() for p in model.parameters())

  size_fp32_mb = (total_params * 4) / (1024 ** 2)
  size_quant_mb = (total_params * (num_bits / 8)) / (1024 ** 2)

  print(f"\nStorage Optimization:")
  print(f"  - Original Size (FP32): {size_fp32_mb:.2f} MB")
  print(f"  - Quantized Size ({num_bits}-bit): {size_quant_mb:.2f} MB")
  print(f"  - Reduction Factor: {size_fp32_mb / size_quant_mb:.1f}x")

model_save_size(model, num_bits=8)

Early Exits Definition

In [ ]:
from typing import List, Optional, Tuple

class ConvNextEarlyExit(nn.Module):
  def __init__(self, base_model, num_classes=2):
    super().__init__()
    self.base_model = base_model
    self.num_classes = num_classes

    self.exit1 = nn.Sequential(
      nn.AdaptiveAvgPool2d((1, 1)),
      nn.Flatten(),
      nn.LazyLinear(num_classes)    #Input shape detected on forward pass
      )

    self.exit2 = nn.Sequential(
      nn.AdaptiveAvgPool2d((1, 1)),
      nn.Flatten(),
      nn.LazyLinear(num_classes)
      )

    self.classifier = base_model.classifier

  def forward(self, pixel_values, labels=None):
    outputs = self.base_model(
      pixel_values=pixel_values,
      output_hidden_states=True,
      return_dict=True
      )

    hidden_states = outputs.hidden_states

    feat_exit1 = hidden_states[-3]  #early exit
    feat_exit2 = hidden_states[-2]  #middle exit

    logits1 = self.exit1(feat_exit1)
    logits2 = self.exit2(feat_exit2)
    logits_final = outputs.logits   #normal exit

    total_loss = None   #loss for new heads training
    losses = {}

    if labels is not None:
      loss_fct = nn.CrossEntropyLoss()
      loss1 = loss_fct(logits1, labels)
      loss2 = loss_fct(logits2, labels)
      loss_final = loss_fct(logits_final, labels)

      total_loss = (0.3 * loss1) + (0.3 * loss2) + (0.4 * loss_final)     #importance score for heads, further it goes likely more "important" it is
      losses = {"exit1": loss1, "exit2": loss2, "final": loss_final}

    return {
      "loss": total_loss,
      "losses": losses,
      "logits_list": [logits1, logits2, logits_final]
    }

Targeted freeze exit head training

In [ ]:
early_exit_model = ConvNextEarlyExit(quantized_model, num_classes=2)
early_exit_model.to("cuda")

for name, param in early_exit_model.base_model.named_parameters():    #param freeze, only train heads, cool hack!
  param.requires_grad = False

print("Layer init...")
dummy_batch = next(iter(train_loader))
dummy_inputs = dummy_batch["pixel_values"].to("cuda")
_ = early_exit_model(dummy_inputs)

optimizer = AdamW(
  filter(lambda p: p.requires_grad, early_exit_model.parameters()), #only exit heads
  lr=1e-3
  )

print("Training Early Exits...")
num_epochs = 3

for epoch in range(num_epochs):
  early_exit_model.train()
  total_loss = 0

  progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

  for batch in progress_bar:
    inputs = batch["pixel_values"].to("cuda")
    labels = batch["label"].to("cuda")

    optimizer.zero_grad()
    outputs = early_exit_model(inputs, labels=labels)

    loss = outputs["loss"]
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    progress_bar.set_postfix(loss=loss.item())

  print(f"Epoch {epoch+1} Average Loss: {total_loss / len(train_loader):.4f}")

print("Early Exit Training Complete!")

Dynamic inference

In [ ]:
import torch.nn.functional as F

def dynamic_inference(model, pixel_values, threshold=0.9):
  backbone = model.base_model.convnext

  x = backbone.embeddings(pixel_values)

  for stage_idx, stage in enumerate(backbone.encoder.stages):
    x = stage(x)

    if stage_idx == 1:          #EXIT 1
      logits = model.exit1(x)
      probs = F.softmax(logits, dim=-1)
      conf, _ = torch.max(probs, dim=-1)
      if conf.item() > threshold:
        return logits, 0

    elif stage_idx == 2:        #EXIT 2
      logits = model.exit2(x)
      probs = F.softmax(logits, dim=-1)
      conf, _ = torch.max(probs, dim=-1)
      if conf.item() > threshold:
        return logits, 1

  x = x.mean(dim=[-2, -1])      #NORMAL EXIT (and flatten to resolve dim error)

  x = backbone.layernorm(x)
  logits = model.classifier(x)

  return logits, 2

In [ ]:
def benchmark_early_exit_full(model, dataset, threshold=0.9):
  print(f"\nBenchmarking Latency & Accuracy (Threshold: {threshold})")

  model.eval()
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model.to(device)

  dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)    #simulate in series inference

  latencies = []
  exit_counts = [0, 0, 0]
  correct_count = 0
  total_count = 0

  print("CUDA sync...")
  dummy = next(iter(dataloader))["pixel_values"].to(device)
  if torch.cuda.is_available(): dummy = dummy.half()
  _ = dynamic_inference(model, dummy, threshold)

  print("Running inference...")

  if torch.cuda.is_available(): torch.cuda.synchronize()    #total set timer
  total_start = time.perf_counter()


  with torch.no_grad():
    for i, batch in enumerate(dataloader):
      # if i >= 1000: break                   #1000 for testing

      inputs = batch["pixel_values"].to(device)
      labels = batch["label"].to(device) # Get the ground truth label

      if torch.cuda.is_available():
        inputs = inputs.half()    #FP32 containers > FP16

      if torch.cuda.is_available(): torch.cuda.synchronize()
      start = time.perf_counter()

      logits, exit_idx = dynamic_inference(model, inputs, threshold)

      if torch.cuda.is_available(): torch.cuda.synchronize()
      end = time.perf_counter()

      latencies.append((end - start) * 1000)
      exit_counts[exit_idx] += 1

      pred = logits.argmax(dim=-1)    #accuracy
      if pred.item() == labels.item():
        correct_count += 1
      total_count += 1

  if torch.cuda.is_available(): torch.cuda.synchronize()
  total_end = time.perf_counter()
  total_time_sec = total_end - total_start

  avg_latency = sum(latencies) / len(latencies)
  accuracy = correct_count / total_count

  print(f"Results:")
  print(f"  - Average Latency: {avg_latency:.2f} ms / image")
  print(f"  - Accuracy:        {accuracy:.2%}")
  print(f"  - Total Set Time:  {total_time_sec:.2f} s")
  print(f"  - Exit Distribution:")
  print(f"    Early Exit:   {exit_counts[0]} ({exit_counts[0]/total_count:.1%})")
  print(f"    Middle Exit: {exit_counts[1]} ({exit_counts[1]/total_count:.1%})")
  print(f"    Normal Exit:  {exit_counts[2]} ({exit_counts[2]/total_count:.1%})")

  return avg_latency, accuracy, exit_counts, total_time_sec

early_exit_model.to("cuda")
early_exit_model.half()       #model quantized but containers are still fp32, need to half it for compatibility
latency, acc, exits, total_time = benchmark_early_exit_full(
  early_exit_model,
  dataset["test"],
  threshold=0.9
  )

##Ablations

In [ ]:
print("--- Baseline (FP32) ---")
baseline_acc, baseline_time = evaluate_model(model, test_loader, device, label="FP32")

In [ ]:
print("--- Baseline (FP32) + Early Exit (0.9 threshold) ---")
latency, acc, exits, total_time = benchmark_early_exit_full(early_exit_model, dataset["test"], threshold=0.9)

In [ ]:
print("--- FP16 ---")
model_fp16 = copy.deepcopy(early_exit_model)
model_fp16.half()
model_fp16.to("cuda")

latency, acc, exits, total_time = benchmark_early_exit_full(model_fp16, dataset["test"], threshold=2.0)     #threshold >1 disables early exits

In [ ]:
print("--- FP16 + Early Exit (0.9 threshold) ---")
latency, acc, exits, total_time = benchmark_early_exit_full(model_fp16, dataset["test"], threshold=0.9)

In [ ]:
print("--- INT8 ---")
model_int8 = copy.deepcopy(early_exit_model)
model_int8.base_model = apply_quantization(model_int8.base_model, num_bits=8)
model_int8.to("cuda")

latency, acc, exits, total_time = benchmark_early_exit_full(model_int8, dataset["test"], threshold=2.0)

In [ ]:
print("--- INT8 + Early Exit (0.9 threshold) ---")
latency, acc, exits, total_time = benchmark_early_exit_full(model_int8, dataset["test"], threshold=0.9)

In [ ]:
print("--- Baseline (FP32) + Early Exit (0.8 threshold) ---")
latency, acc, exits, total_time = benchmark_early_exit_full(early_exit_model, dataset["test"], threshold=0.8)

In [ ]:
print("--- FP16 + Early Exit (0.8 threshold) ---")
latency, acc, exits, total_time = benchmark_early_exit_full(model_fp16, dataset["test"], threshold=0.8)

In [ ]:
print("--- INT8 + Early Exit (0.8 threshold) ---")
latency, acc, exits, total_time = benchmark_early_exit_full(model_int8, dataset["test"], threshold=0.8)

##Visualizations

In [ ]:
ablations = [
  "FP32", "FP32 + 0.9 Exit", "FP32 + 0.8 Exit",
  "INT8", "INT8 + 0.9 Exit", "INT8 + 0.8 Exit",
  "FP16", "FP16 + 0.9 Exit", "FP16 + 0.8 Exit"
]

latency_vals =  [11.42, 5.04, 4.23, 11.07, 4.96, 4.24, 11.23, 5.05, 4.25]
accuracy_vals = [84.26, 79.97, 81.34, 86.41, 80.11, 79.97, 83.94, 79.97, 81.13]

groups = ["FP32", "INT8 ", "FP16"]
colors = ['gray', 'red', 'blue']
markers = ['o', '^', 'v']
threshold_labels = ["Normal Exit", "t=0.9", "t=0.8"]

text_offsets = [
  [(0, 10), (0, 15), (0, 15)],
  [(0, -15), (15, -10), (15, -10)],
  [(3, -15), (-25, 0), (-25, 0)]
  ]

plt.figure(figsize=(12, 8))

for group_idx in range(3):
  start_idx = group_idx * 3
  end_idx = start_idx + 3

  group_l = latency_vals[start_idx:end_idx]
  group_a = accuracy_vals[start_idx:end_idx]
  color = colors[group_idx]

  plt.plot(group_l, group_a, color=color, linestyle='--', alpha=0.4, label=groups[group_idx])

  for i in range(3):
    plt.scatter(group_l[i], group_a[i],
      color=color, marker=markers[i],
      s=150 if i==0 else 120,
      edgecolors='black', alpha=0.8, zorder=10)

    offset = text_offsets[group_idx][i]

    if group_idx == 0 or i == 0:
      label_txt = threshold_labels[i]

      plt.annotate(label_txt, (group_l[i], group_a[i]),
        xytext=offset, textcoords='offset points',
        fontsize=10, color=color, fontweight='bold')

plt.title("Pareto Frontier", fontsize=16, fontweight='bold')
plt.xlabel("Inference Latency (ms)", fontsize=13)
plt.ylabel("Test Accuracy (%)", fontsize=13)

plt.xlim(4, max(latency_vals) * 1.2)
plt.ylim(78, 88)

plt.grid(True, linestyle=':', alpha=0.6)

from matplotlib.lines import Line2D
legend_elements = [
  Line2D([0], [0], color='gray', lw=2, linestyle='--', label='FP32'),
  Line2D([0], [0], color='red', lw=2, linestyle='--', label='INT8'),
  Line2D([0], [0], color='blue', lw=2, linestyle='--', label='FP16'),
  Line2D([0], [0], marker='o', color='w', markerfacecolor='k', markersize=10, label='Baseline (Normal Exit)'),
  Line2D([0], [0], marker='^', color='w', markerfacecolor='k', markersize=10, label='Threshold 0.9'),
  Line2D([0], [0], marker='v', color='w', markerfacecolor='k', markersize=10, label='Threshold 0.8'),
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.show()

Total test set inference time graph:

In [ ]:
ablations = [
  "FP32", "FP32 + 0.9 Exit", "FP32 + 0.8 Exit",
  "INT8", "INT8 + 0.9 Exit", "INT8 + 0.8 Exit",
  "FP16", "FP16 + 0.9 Exit", "FP16 + 0.8 Exit"
]
total_set_times = [26, 20.92, 20.49, 25.11, 21.02, 20.46, 24.89, 21.09, 20.67]

df = pd.DataFrame({
  "Ablation Configuration": ablations,
  "Total Time (s)": total_set_times
})

def get_precision(name):
    if "INT8" in name: return "INT8"
    if "FP16" in name: return "FP16"
    return "FP32"

df['Precision'] = df['Ablation Configuration'].apply(get_precision)

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

barplot = sns.barplot(
  data=df,
  x="Ablation Configuration",
  y="Total Time (s)",
  hue="Precision",
  palette="viridis",
  dodge=False
  )

for i, v in enumerate(total_set_times):
  plt.text(i, v + 0.2, f"{v}s", ha='center', va='bottom', fontweight='bold')

plt.title("Total Inference Time on Test Set", fontsize=16, fontweight='bold')
plt.xlabel("Configuration", fontsize=12)
plt.ylabel("Total Time (Seconds)", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 30)

plt.tight_layout()
plt.show()

Model size reduction graph:

In [ ]:
precisions = ['FP32 (Baseline)', 'FP16', 'INT8']
sizes = [748.57, 374.29, 187.14]
sns.set_style("whitegrid")
plt.figure(figsize=(8, 6))

barplot = sns.barplot(x=precisions, y=sizes, palette="Blues_r")

for i, v in enumerate(sizes):
  barplot.text(i, v + 15, f"{v} MB", ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.title("Model Size Reduction via Quantization", fontsize=16, fontweight='bold')
plt.ylabel("Model Size (MB)", fontsize=12)
plt.xlabel("Precision", fontsize=12)
plt.ylim(0, 900)

plt.tight_layout()
plt.savefig("quantization_size_reduction.png")
plt.show()

INT8 result unexpected but understandable - the quantization is acting as a form of regressions and is allowing better generalization, avoiding overfitting. One would assume INT8 is too aggressive for this but it seems not.

Results table:

| Configuration | Accuracy (%) | Latency (ms) |
| :--- | :--- | :--- |
| FP32 | 84.26 | 11.42 |
| FP32 + 0.9 Exit | 79.97 | 5.04 |
| FP32 + 0.8 Exit | 81.34 | 4.23 |
| | | |
| FP16 | 83.94 | 11.23 |
| FP16 + 0.9 Exit | 79.97 | 5.05 |
| FP16 + 0.8 Exit | 81.13 | 4.25 |
| | | |
| INT8 | 86.41 | 11.07 |
| INT8 + 0.9 Exit | 80.11 | 4.96 |
| INT8 + 0.8 Exit | 79.97 | 4.24 |

Compute save graph:

In [ ]:
exits = ["Early Exit 1", "Early Exit 2", "Normal Exit"]
gflops = [4.5, 20.0, 34.4]

sns.set_style("whitegrid")
plt.figure(figsize=(8, 6))

barplot = sns.barplot(x=exits, y=gflops, palette="viridis")

for i, v in enumerate(gflops):
  plt.text(i, v + 0.5, f"{v} GFLOPs", ha='center', va='bottom', fontweight='bold', color='black')

plt.title("X-Pedite Computational Cost", fontsize=16, fontweight='bold')
plt.xlabel("Exit Stage", fontsize=12)
plt.ylabel("Computational Cost (GFLOPs)", fontsize=12)
plt.ylim(0, 40)

plt.tight_layout()
plt.show()

(Sanity check) Activation heatmap:

In [ ]:
!pip install grad-cam

In [ ]:
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image

class HFWrapper(torch.nn.Module):
  def __init__(self, model):
    super(HFWrapper, self).__init__()
    self.model = model

  def forward(self, x):
    return self.model(x).logits

model_wrapper = HFWrapper(model)
target_layers = [model.convnext.encoder.stages[-1].layers[-1]]

image_path = "/content/person92_virus_174.jpeg"
img = np.array(Image.open(image_path).convert('RGB'))
rgb_img = np.float32(img) / 255.0
input_tensor = preprocess_image(rgb_img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

cam = GradCAM(model=model_wrapper, target_layers=target_layers)
grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]

visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(rgb_img)
plt.title("Original X-Ray")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(visualization)
plt.title("Model Attention")
plt.axis("off")

plt.tight_layout()
plt.show()

NOTE: Need to manually find and import image from local for above script (chest-xrays/train/PNEUMONIA)